In [5]:
# Task 1:  Implement quantization from scratch

def quantize_int8(values):

    q_min = -128
    q_max = 127

    # find minimum and maximum
    min_val = min(values)
    max_val = max(values)

    # Calculate scale
    scale = (max_val - min_val) / (q_max - q_min)

    # Calculate zero-point
    zero_point = round(q_min - (min_val / scale))

    quantized = []

    # Quantize each value
    for x in values:
        q = round(x / scale + zero_point)

        q = max(q_min, min(q_max, q))

        quantized.append(q)

    return quantized, scale, zero_point


# array size
size = int(input("Enter the size of the floating array: "))

floatList = []

# take values one by one
for i in range(size):
    num = float(input(f"Enter number {i + 1}: "))
    floatList.append(num)

values = floatList

# Quantize the values
quantized_values, scale, zero_point = quantize_int8(values)

# Display results
print("Original values:")
print(values)

print("Quantized values:")
print(quantized_values)

print("Scale:", scale)
print("Zero Point:", zero_point)

Enter the size of the floating array: 7
Enter number 1: -10.5 
Enter number 2: -5.2
Enter number 3: -1.8
Enter number 4: 0
Enter number 5: 3.4
Enter number 6: 7.6
Enter number 7: 10.2
Original values:
[-10.5, -5.2, -1.8, 0.0, 3.4, 7.6, 10.2]
Quantized values:
[-128, -63, -21, 1, 43, 95, 127]
Scale: 0.0811764705882353
Zero Point: 1


The FP32 numbers were converted into int8, which helps reduce the memory size required to store the data.

In [9]:
# Task 2: Estimate Model Size from Parameter Count

def model_size(parameters, dtype):

    # Bytes used by each parameter
    if dtype == "fp32":
        bytes_per_parameter = 4

    elif dtype == "fp16":
        bytes_per_parameter = 2

    elif dtype == "int8":
        bytes_per_parameter = 1

    else:
        print("Invalid data type")
        return

    # Calculate size in bytes
    size_bytes = parameters * bytes_per_parameter

    # mb and gb
    size_mb = size_bytes / (1024 ** 2)
    size_gb = size_bytes / (1024 ** 3)

    return size_mb, size_gb


# input
parameters = int(input("Enter number of parameters: "))
dtype = input("Enter data type (fp32/fp16/int8): ").lower()

# Calculate model size
size_mb, size_gb = model_size(parameters, dtype)

# Display result
print("\nModel Size:")
print("Size in MB:", size_mb)
print("Size in GB:", size_gb)

Enter number of parameters: 125000000
Enter data type (fp32/fp16/int8): fp32

Model Size:
Size in MB: 476.837158203125
Size in GB: 0.46566128730773926


In [ ]:
#outputs :
#Enter number of parameters: 1000000000
#Enter data type (fp32/fp16/int8): int8

#Model Size:
#Size in MB: 953.67431640625
#Size in GB: 0.9313225746154785

#Enter number of parameters: 7000000000
#Enter data type (fp32/fp16/int8): fp16

#Model Size:
#Size in MB: 13351.4404296875
#Size in GB: 13.0385160446167


As the precision decreases from FP32 to INT8, the model size decreases significantly. INT8 requires only 1 byte per parameter compared with 4 bytes for FP32.

Task 3 System Design Scenario

Where would you split the model (within a node vs across nodes), and why?

First, I will split the model across the 8 GPUs within the same node using NVLink and stilll if the model doesn't fit , split the remaining parts across multiple nodes using InfiniBand.


What role does NVLink play vs InfiniBand in this setup?

According to me ,NVLink are used for GPU-to-GPU communication within the same node. It is useful because different GPUs need to exchange activations and intermediate results during inference whereas InfiniBand provides high-speed communication between different nodes. It allows GPUs on separate servers to communicate when the model needs to span multiple machines.


What would go wrong if you swapped their roles?

If we use InfiniBand for GPUs within the same node, communication can become less efficient than using the high-speed GPU interconnect.
If we try to use NVLink across separate nodes, it generally cannot serve as the normal cross-node network connection; NVLink is primarily designed for GPU-to-GPU connectivity within supported systems.


Write your answer as a short design note (half a page), no code needed.

Since the model cannot fit on a single GPU, the model should be divided across multiple GPUs. Within a node, the model should preferably be split across the 8 GPUs connected through NVLink because it provides fast GPU-to-GPU communication and reduces communication overhead. If the model is still too large for one node, additional nodes can be used. InfiniBand is used for communication between GPUs located on different nodes. NVLink is therefore mainly used for fast gpu communication, while InfiniBand handles inter-node communication. Using the appropriate interconnect helps reduce communication latency and improves inference performance.

In [10]:
# Task 4: Compute-bound vs Memory-bound

def check_bound(flops, compute_throughput, memory_bytes, memory_bandwidth):

    # Calculate compute time
    compute_time = flops / compute_throughput

    # Calculate memory time
    memory_time = memory_bytes / memory_bandwidth

    if compute_time > memory_time:
        result = "Compute-bound"
    else:
        result = "Memory-bound"

    return result, compute_time, memory_time


# Test 1
flops = 100_000_000_000
compute_throughput = 100_000_000_000_000
memory_bytes = 1_000_000_000
memory_bandwidth = 1_000_000_000_000

result, compute_time, memory_time = check_bound(
    flops, compute_throughput, memory_bytes, memory_bandwidth
)

print("Test 1")
print("Compute time:", compute_time, "seconds")
print("Memory time:", memory_time, "seconds")
print("Result:", result)


# Test 2
flops = 500_000_000_000
compute_throughput = 100_000_000_000_000
memory_bytes = 5_000_000_000
memory_bandwidth = 1_000_000_000_000

result, compute_time, memory_time = check_bound(
    flops, compute_throughput, memory_bytes, memory_bandwidth
)

print("\nTest 2")
print("Compute time:", compute_time, "seconds")
print("Memory time:", memory_time, "seconds")
print("Result:", result)


# Test 3
flops = 800_000_000_000
compute_throughput = 100_000_000_000_000
memory_bytes = 500_000_000
memory_bandwidth = 1_000_000_000_000

result, compute_time, memory_time = check_bound(
    flops, compute_throughput, memory_bytes, memory_bandwidth
)

print("\nTest 3")
print("Compute time:", compute_time, "seconds")
print("Memory time:", memory_time, "seconds")
print("Result:", result)

Test 1
Compute time: 0.001 seconds
Memory time: 0.001 seconds
Result: Memory-bound

Test 2
Compute time: 0.005 seconds
Memory time: 0.005 seconds
Result: Memory-bound

Test 3
Compute time: 0.008 seconds
Memory time: 0.0005 seconds
Result: Compute-bound


When compute time is higher, the workload is compute-bound; when memory time is higher, it is memory-bound.Compute-bound workloads benefit from better compute throughput, while memory-bound workloads benefit from higher memory bandwidth or reduced memory traffic.